In [ ]:
!pip install -q "huggingface_hub>=1.3,<2" "polars>=1.32,<2" "pyarrow>=18,<23" "tokenizers>=0.21" "tqdm>=4.66"


In [ ]:
from decimal import Decimal
from pathlib import Path
from time import perf_counter
import contextlib
import io
import json
import math
import numbers
import signal
import sys

import polars as pl
import torch
from huggingface_hub import hf_hub_download
from IPython.display import FileLink, display
from tokenizers import Tokenizer
from torch import nn
from torch.nn import functional as F
from tqdm.auto import tqdm

if Path("/kaggle/working").is_dir():
    WORK = Path("/kaggle/working/inference_100k")
elif Path("/content").is_dir():
    WORK = Path("/content/inference_100k")
else:
    WORK = Path.cwd() / "inference_100k"
WORK.mkdir(parents=True, exist_ok=True)

PUBLIC_REPO = "kryalka/nlp-code-generation-100k"
PUBLIC_REVISION = "85f308df5ef6f28191210003b57f0874acfc6858"
CHECKPOINT_FILE = "model_step_100000.pt"
TOKENIZER_FILE = "tokenizer.json"
TEST_FILE = "test.parquet"
BEAM_SIZE = 16
BATCH_SIZE = 64
MAX_NEW_TOKENS = 160
LENGTH_PENALTY = 0.6
RERANK_TEMPERATURE = 1.0
FALLBACK_MAX_NEW_TOKENS = 256
OUTPUT = WORK / "prediction.jsonl"

checkpoint_path = hf_hub_download(PUBLIC_REPO, CHECKPOINT_FILE, revision=PUBLIC_REVISION)
tokenizer_path = hf_hub_download(PUBLIC_REPO, TOKENIZER_FILE, revision=PUBLIC_REVISION)
test_path = hf_hub_download(PUBLIC_REPO, TEST_FILE, revision=PUBLIC_REVISION)
print("найс, файлы модели и данных загружены")


In [ ]:
class Attention(nn.Module):
    def __init__(self, d_model, num_heads, dropout):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.dropout = dropout
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        b, length, _ = x.shape
        return x.view(b, length, self.num_heads, self.head_dim).transpose(1, 2)

    def project_kv(self, x):
        return self.split_heads(self.k(x)), self.split_heads(self.v(x))

    def forward(self, x, context=None, mask=None, causal=False, cache=None, static_kv=None):
        q = self.split_heads(self.q(x))
        if static_kv is not None:
            k, v = static_kv
        else:
            k, v = self.project_kv(x if context is None else context)
            if cache is not None:
                k = torch.cat((cache[0], k), dim=2)
                v = torch.cat((cache[1], v), dim=2)

        result = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=causal,
        )
        result = result.transpose(1, 2).contiguous().view(x.size(0), x.size(1), -1)
        return self.out(result), (k, v)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(d_model, d_ff), 
            nn.ReLU(), 
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.layers(x)


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = Attention(d_model, num_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        y, _ = self.self_attn(self.norm1(x), mask=src_mask)
        x = x + self.drop(y)
        return x + self.drop(self.ff(self.norm2(x)))


class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = Attention(d_model, num_heads, dropout)
        self.cross_attn = Attention(d_model, num_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, memory, src_mask):
        y, _ = self.self_attn(self.norm1(x), causal=True)
        x = x + self.drop(y)
        y, _ = self.cross_attn(self.norm2(x), context=memory, mask=src_mask)
        x = x + self.drop(y)
        return x + self.drop(self.ff(self.norm3(x)))

    def step(self, x, src_mask, self_cache, cross_kv):
        y, new_cache = self.self_attn(self.norm1(x), cache=self_cache)
        x = x + self.drop(y)
        y, _ = self.cross_attn(self.norm2(x), mask=src_mask, static_kv=cross_kv)
        x = x + self.drop(y)
        x = x + self.drop(self.ff(self.norm3(x)))
        return x, new_cache


class Transformer(nn.Module):
    def __init__(self, cfg, pad_id=0, eos_id=1):
        super().__init__()
        d = cfg["d_model"]
        self.pad_id, self.eos_id = pad_id, eos_id
        self.max_source_length = cfg["max_source_length"]
        self.max_target_length = cfg["max_target_length"]
      
        self.embedding = nn.Embedding(cfg["vocab_size"], d)
        self.src_pos = nn.Embedding(self.max_source_length, d)
        self.tgt_pos = nn.Embedding(self.max_target_length, d)
        self.drop = nn.Dropout(cfg["dropout"])
        self.encoder = nn.ModuleList([
            EncoderLayer(d, cfg["num_heads"], cfg["d_ff"], cfg["dropout"])
            for _ in range(cfg["encoder_layers"])
        ])
        self.decoder = nn.ModuleList([
            DecoderLayer(d, cfg["num_heads"], cfg["d_ff"], cfg["dropout"])
            for _ in range(cfg["decoder_layers"])
        ])
        self.enc_norm = nn.LayerNorm(d)
        self.dec_norm = nn.LayerNorm(d)

        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def source_mask(self, source):
        return (source != self.pad_id)[:, None, None, :]

    def encode(self, source):
        b, length = source.shape
        assert length <= self.max_source_length
        positions = torch.arange(length, device=source.device)[None, :]
        x = self.embedding(source)
        x = x + self.src_pos(positions)
        x = self.drop(x)
        mask = self.source_mask(source)
        for layer in self.encoder:
            x = layer(x, mask)
        return self.enc_norm(x), mask

    def forward(self, source, target_input, labels=None):
        memory, mask = self.encode(source)
        length = target_input.size(1)
        assert length <= self.max_target_length
        positions = torch.arange(length, device=target_input.device)[None, :]
        x = self.embedding(target_input)
        x = x + self.tgt_pos(positions)
        x = self.drop(x)
        for layer in self.decoder:
            x = layer(x, memory, mask)
        x = self.dec_norm(x)
        if labels is not None:
            x = x[labels.ne(-100)]
        return F.linear(x, self.embedding.weight)

    @torch.inference_mode()
    def generate(self, source, max_new_tokens=160, temperature=0.0, top_p=1.0,
                 num_candidates=1, return_scores=False, generator=None):
        memory, mask = self.encode(source)
        cross_kv = [layer.cross_attn.project_kv(memory) for layer in self.decoder]
        if num_candidates > 1:
            mask = mask.repeat_interleave(num_candidates, dim=0)
            cross_kv = [(k.repeat_interleave(num_candidates, dim=0),
                         v.repeat_interleave(num_candidates, dim=0)) for k, v in cross_kv]
        self_cache = [None] * len(self.decoder)
        batch_size = source.size(0) * num_candidates
        token = torch.full((batch_size, 1), self.pad_id, dtype=torch.long, device=source.device)
        finished = torch.zeros(batch_size, dtype=torch.bool, device=source.device)
        outputs = []
        if return_scores:
            logprob = torch.zeros(batch_size, device=source.device)
            lengths = torch.zeros(batch_size, dtype=torch.long, device=source.device)

        for position in range(max_new_tokens):
            pos = torch.full_like(token, position)
            x = self.embedding(token)
            x = x + self.tgt_pos(pos)
            x = self.drop(x)
            for i, layer in enumerate(self.decoder):
                x, self_cache[i] = layer.step(x, mask, self_cache[i], cross_kv[i])
            logits = F.linear(self.dec_norm(x[:, -1]), self.embedding.weight)
            if temperature == 0:
                next_token = logits.argmax(dim=-1)
            else:
                probabilities = F.softmax(logits.float() / temperature, dim=-1)
                probabilities, indices = probabilities.sort(dim=-1, descending=True)
                remove = probabilities.cumsum(dim=-1) - probabilities >= top_p
                probabilities.masked_fill_(remove, 0)
                sampled = torch.multinomial(probabilities, 1, generator=generator)
                next_token = indices.gather(1, sampled).squeeze(1)
            if return_scores:
                scores = F.log_softmax(logits.float(), dim=-1).gather(1, next_token[:, None]).squeeze(1)
                logprob += scores.masked_fill(finished, 0)
                lengths += ~finished
            next_token = torch.where(finished, self.pad_id, next_token)
            outputs.append(next_token)
            finished |= next_token.eq(self.eos_id)
            if bool(finished.all()):
                break
            token = next_token[:, None]

        generated = torch.stack(outputs, dim=1)
        if return_scores:
            return generated, logprob, lengths
        return generated



In [ ]:
@torch.inference_mode()
def beam_generate(model, source, max_new_tokens=160, beam_size=4, length_penalty=0.6, return_all=False):
    memory, mask = model.encode(source)
    batch = source.size(0)
    cross_kv = [layer.cross_attn.project_kv(memory) for layer in model.decoder]
    cross_kv = [
        (key.repeat_interleave(beam_size, 0), value.repeat_interleave(beam_size, 0))
        for key, value in cross_kv
    ]
    mask = mask.repeat_interleave(beam_size, 0)
    cache = [None] * len(model.decoder)
    token = torch.full(
        (batch * beam_size, 1), model.pad_id, dtype=torch.long, device=source.device
    )
    scores = torch.full((batch, beam_size), -torch.inf, device=source.device)
    scores[:, 0] = 0.0
    finished = torch.zeros((batch, beam_size), dtype=torch.bool, device=source.device)
    lengths = torch.zeros((batch, beam_size), dtype=torch.long, device=source.device)
    sequences = torch.empty((batch, beam_size, 0), dtype=torch.long, device=source.device)

    for position in range(max_new_tokens):
        pos = torch.full_like(token, position)
        hidden = model.embedding(token) + model.tgt_pos(pos)
        hidden = model.drop(hidden)
        new_cache = []
        for index, layer in enumerate(model.decoder):
            hidden, layer_cache = layer.step(hidden, mask, cache[index], cross_kv[index])
            new_cache.append(layer_cache)
        logits = F.linear(model.dec_norm(hidden[:, -1]), model.embedding.weight)
        log_probs = F.log_softmax(logits.float(), dim=-1).view(batch, beam_size, -1)

        forced = torch.full_like(log_probs, -torch.inf)
        forced[:, :, model.pad_id] = 0.0
        log_probs = torch.where(finished.unsqueeze(-1), forced, log_probs)
        candidates = scores.unsqueeze(-1) + log_probs
        scores, indices = candidates.view(batch, -1).topk(beam_size, dim=-1)
        parents = indices // logits.size(-1)
        next_tokens = indices % logits.size(-1)

        gather_seq = parents.unsqueeze(-1).expand(-1, -1, sequences.size(-1))
        sequences = sequences.gather(1, gather_seq)
        parent_finished = finished.gather(1, parents)
        lengths = lengths.gather(1, parents) + (~parent_finished).long()
        finished = parent_finished | next_tokens.eq(model.eos_id)
        sequences = torch.cat((sequences, next_tokens.unsqueeze(-1)), dim=-1)

        flat_parents = (
            torch.arange(batch, device=source.device).unsqueeze(1) * beam_size + parents
        ).reshape(-1)
        cache = [
            (key.index_select(0, flat_parents), value.index_select(0, flat_parents))
            for key, value in new_cache
        ]
        token = next_tokens.reshape(-1, 1)
        if bool(finished.all()):
            break

    penalty = ((5.0 + lengths.clamp_min(1).float()) / 6.0).pow(length_penalty)
    normalized = scores / penalty
    order = normalized.argsort(dim=-1, descending=True)
    sequences = sequences.gather(1, order.unsqueeze(-1).expand_as(sequences))
    normalized = normalized.gather(1, order)
    if return_all:
        return sequences, normalized
    return sequences[:, 0]


def decode_rows(tokenizer, rows):
    result = []
    for row in rows.tolist():
        if 1 in row:
            row = row[:row.index(1)]
        result.append(tokenizer.decode(row, skip_special_tokens=True))
    return result

def prepare_sources(questions):
    rows = [item.ids for item in tokenizer.encode_batch(questions)]
    limit = model_cfg["max_source_length"]
    rows = [row if len(row) <= limit else row[:limit - 1] + [1] for row in rows]
    width = max(map(len, rows))
    return torch.tensor(
        [row + [0] * (width - len(row)) for row in rows],
        dtype=torch.long, device=device,
    )


class ExecutionTimeout(Exception):
    pass

def timeout_handler(signum, frame):
    raise ExecutionTimeout

signal.signal(signal.SIGALRM, timeout_handler)

def run_code_fast(program, timeout=0.05):
    namespace = {"__name__": "__generated__"}
    try:
        signal.setitimer(signal.ITIMER_REAL, timeout)
        with contextlib.redirect_stdout(io.StringIO()):
            exec(program, namespace)
            value = namespace["solve"]()
        numeric = isinstance(value, (numbers.Real, Decimal)) and not isinstance(value, bool)
        if not numeric and "sympy" in sys.modules:
            numeric = isinstance(value, sys.modules["sympy"].Number) and value.is_real is True
        value = float(value) if numeric else None
        return value if value is not None and math.isfinite(value) else None
    except BaseException:
        return None
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)

def same_result(a, b):
    return a is not None and b is not None and abs(a - b) <= 1e-6 + 1e-6 * abs(b)

def weighted_choice(results, scores):
    valid = [index for index, value in enumerate(results) if value is not None]
    if not valid:
        return 0
    groups = []
    for index in valid:
        for group in groups:
            if same_result(results[index], results[group[0]]):
                group.append(index)
                break
        else:
            groups.append([index])
    maximum = max(scores[index] for index in valid)
    def weight(group):
        return sum(math.exp((scores[index] - maximum) / RERANK_TEMPERATURE) for index in group)
    return max(groups, key=weight)[0]



In [ ]:
device = torch.device("cuda")
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
model_cfg = checkpoint["config"]
model = Transformer(model_cfg).to(device)
model.load_state_dict(checkpoint["model"])
model.eval()
tokenizer = Tokenizer.from_file(str(tokenizer_path))
questions = pl.read_parquet(test_path, columns=["question_clean"])["question_clean"].to_list()
print("GPU:", torch.cuda.get_device_name(0))
print("test:", len(questions), "beam:", BEAM_SIZE, "batch:", BATCH_SIZE)


In [ ]:
started = perf_counter()
with OUTPUT.open("w", encoding="utf-8") as output:
    for start in tqdm(range(0, len(questions), BATCH_SIZE), desc="Beam search + rerank"):
        source = prepare_sources(questions[start:start + BATCH_SIZE])
        with torch.autocast("cuda", dtype=torch.float16):
            sequences, scores = beam_generate(
                model, source, MAX_NEW_TOKENS, BEAM_SIZE,
                LENGTH_PENALTY, return_all=True,
            )
        scores = scores.cpu().tolist()
        for local_index, (sample, sample_scores) in enumerate(zip(sequences, scores)):
            candidates = decode_rows(tokenizer, sample)
            results = [run_code_fast(program) for program in candidates]
            if not any(value is not None for value in results):
                with torch.autocast("cuda", dtype=torch.float16):
                    fallback_sequences, fallback_scores = beam_generate(
                        model, source[local_index:local_index + 1],
                        FALLBACK_MAX_NEW_TOKENS, BEAM_SIZE,
                        LENGTH_PENALTY, return_all=True,
                    )
                candidates = decode_rows(tokenizer, fallback_sequences[0])
                sample_scores = fallback_scores[0].cpu().tolist()
                results = [run_code_fast(program) for program in candidates]
            selected = weighted_choice(results, sample_scores)
            output.write(json.dumps({"code": candidates[selected]}, ensure_ascii=False) + "\n")

elapsed = perf_counter() - started
print(f"Сохранено: {OUTPUT}; строк: {len(questions)}; время: {elapsed:.2f} с")


In [ ]:
try:
    from google.colab import files
    files.download(str(OUTPUT))
except ImportError:
    display(FileLink(str(OUTPUT)))
    print(f"Файл сохранён: {OUTPUT}")
